# Ground Truth generation with LightOnOCR-2-1B

Extracts tables from scientific paper PDFs and emits a JSON with the same structure as `ground_truth_kge.json`.

- Model: `lightonai/LightOnOCR-2-1B` (1B-param end-to-end OCR VLM)
- Input: PDF pages rendered as images (200 DPI, longest side ≤ 1540px)
- Output: HTML/Markdown tables → parsed into structured JSON

In [ ]:
# Do NOT reinstall torch/torchvision on the GPU server to avoid CUDA conflicts;
# only install transformers v5+ (from git) and lightweight deps.
%pip install -q -U "transformers @ git+https://github.com/huggingface/transformers.git" pillow pypdfium2 accelerate beautifulsoup4

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import re
import base64
import io
from pathlib import Path

import torch
import pypdfium2 as pdfium
from PIL import Image
from bs4 import BeautifulSoup
from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor

In [ ]:
# Environment check
import torch, transformers, sys
print("Python:           ", sys.version.split()[0])
print("torch:            ", torch.__version__)
print("torch.cuda avail: ", torch.cuda.is_available())
print("CUDA build:       ", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:              ", torch.cuda.get_device_name(0))
print("transformers:     ", transformers.__version__)

try:
    from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor
    print("LightOnOcr classes: OK")
except ImportError as e:
    print("LightOnOcr classes: FAILED →", e)

from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
print("lighton_ocr in CONFIG_MAPPING:", "lighton_ocr" in CONFIG_MAPPING_NAMES)

In [ ]:
# Paths — set PDF_DIR to your folder of papers
PDF_DIR = Path("pdfs_test")
OUTPUT_PATH = PDF_DIR / "ground_truth" / "ground_truth_lightonocr.json"

if not PDF_DIR.is_dir():
    raise FileNotFoundError(f"Missing folder: {PDF_DIR.resolve()}")

PDF_FILES = sorted(PDF_DIR.glob("*.pdf"))
if not PDF_FILES:
    raise FileNotFoundError(f"No PDFs found in {PDF_DIR.resolve()}")

print(f"PDF_DIR: {PDF_DIR.resolve()}")
print(f"OUTPUT_PATH: {OUTPUT_PATH.resolve()}")
print(f"PDFs found: {len(PDF_FILES)}")
for p in PDF_FILES:
    print(f"  - {p.name}")

## Load the LightOnOCR-2-1B model

In [ ]:
MODEL_ID = "lightonai/LightOnOCR-2-1B"

# Auto-detect device
if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float32
else:
    device = "cpu"
    dtype = torch.float32

print(f"Device: {device}, dtype: {dtype}")

processor = LightOnOcrProcessor.from_pretrained(MODEL_ID)
model = LightOnOcrForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    attn_implementation="eager",
).to(device)

print(f"Model loaded: {MODEL_ID}")
print(f"Processor: {type(processor).__name__}")
print(f"Model: {type(model).__name__}")

## Helper functions

In [ ]:
def render_pdf_page(pdf_doc, page_idx: int, target_longest: int = 1540) -> Image.Image:
    """Render a PDF page as a PIL image at 200 DPI, resized so the longest side is
    target_longest px (recommended by the LightOnOCR model card)."""
    page = pdf_doc[page_idx]
    bitmap = page.render(scale=200 / 72)
    pil_image = bitmap.to_pil()

    w, h = pil_image.size
    longest = max(w, h)
    if longest > target_longest:
        ratio = target_longest / longest
        pil_image = pil_image.resize(
            (int(w * ratio), int(h * ratio)), Image.LANCZOS
        )

    if pil_image.mode != "RGB":
        pil_image = pil_image.convert("RGB")

    return pil_image


def ocr_page(pil_image: Image.Image, max_new_tokens: int = 8192) -> str:
    """Run LightOnOCR on a page image and return the generated text (HTML/markdown)."""
    import tempfile, os
    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    pil_image.save(tmp, format="PNG")
    tmp.close()

    conversation = [
        {"role": "user", "content": [{"type": "image", "url": tmp.name}]}
    ]

    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {
        k: v.to(device=device, dtype=dtype) if v.is_floating_point() else v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    os.unlink(tmp.name)
    return processor.decode(generated_ids, skip_special_tokens=True)


print("Rendering and OCR functions loaded.")

In [ ]:
def coerce_value(s: str):
    """Coerce a string to int/float when possible; return '-' for missing values.
    Handles thousand separators (e.g. '40,943' → 40943). Kept consistent with the
    DeepDoctection notebook."""
    s = s.strip()
    if not s or s in ('–', '—', 'N/A', 'n/a', 'nan', ''):
        return '-'
    if s == '-':
        return '-'

    s_clean = s
    if re.search(r',\d{3}(\D|$)', s):
        s_clean = s.replace(',', '')
    else:
        s_clean = s.replace(',', '.')

    try:
        val = float(s_clean)
        return int(val) if val == int(val) else val
    except ValueError:
        return s


def clean_cell_text(text: str) -> str:
    """Strip residual markdown/LaTeX formatting from a cell value."""
    text = text.strip()
    # Strip bold/italic
    text = re.sub(r'\*\*(.+?)\*\*', r'\1', text)
    text = re.sub(r'(?<!\*)\*(?!\*)(.+?)(?<!\*)\*(?!\*)', r'\1', text)
    # Strip inline code
    text = re.sub(r'`(.+?)`', r'\1', text)
    # Strip inline math
    text = re.sub(r'\$([^$]+?)\$', r'\1', text)
    # Strip common LaTeX commands
    text = re.sub(r'\\textbf\{(.+?)\}', r'\1', text)
    text = re.sub(r'\\textit\{(.+?)\}', r'\1', text)
    text = re.sub(r'\\text\{(.+?)\}', r'\1', text)
    text = re.sub(r'\\mathbf\{(.+?)\}', r'\1', text)
    return text.strip()


def is_separator_line(line: str) -> bool:
    """Return True if the line is a markdown table separator (|---|---|)."""
    cells = [c.strip() for c in line.strip().split('|')]
    cells = [c for c in cells if c]
    if not cells:
        return False
    return all(re.match(r'^:?-{1,}:?$', c) for c in cells)


def extract_markdown_tables(text: str) -> list:
    """Extract markdown table blocks from a page's OCR output.
    A valid block has consecutive '|...|' lines, at least one '|---|' separator
    and >=3 lines total (header + separator + >=1 data row)."""
    lines = text.split('\n')
    tables = []
    current_block = []
    in_table = False

    for line in lines:
        stripped = line.strip()
        if stripped.startswith('|') and '|' in stripped[1:]:
            current_block.append(stripped)
            in_table = True
        else:
            if in_table and current_block:
                has_sep = any(is_separator_line(l) for l in current_block)
                if has_sep and len(current_block) >= 3:
                    tables.append(current_block[:])
                current_block = []
                in_table = False

    # Flush final block
    if current_block:
        has_sep = any(is_separator_line(l) for l in current_block)
        if has_sep and len(current_block) >= 3:
            tables.append(current_block[:])

    return tables


def parse_row_cells(line: str) -> list:
    """Parse a markdown table row into individual cells."""
    cells = line.split('|')
    if cells and cells[0].strip() == '':
        cells = cells[1:]
    if cells and cells[-1].strip() == '':
        cells = cells[:-1]
    return [clean_cell_text(c) for c in cells]


def parse_markdown_table(table_lines: list) -> tuple:
    """Parse a markdown table block into (columns, data_rows).
    Detects multi-level headers (rows before the separator), flattens them with
    underscores (matching the manual GT), and coerces values via coerce_value."""
    # Find separator line
    sep_idx = None
    for i, line in enumerate(table_lines):
        if is_separator_line(line):
            sep_idx = i
            break

    if sep_idx is None or sep_idx == 0:
        return None, None

    header_lines = table_lines[:sep_idx]
    data_lines = table_lines[sep_idx + 1:]

    if not data_lines:
        return None, None

    # Parse header rows
    header_rows = [parse_row_cells(line) for line in header_lines]

    # Column count
    all_rows_parsed = header_rows + [parse_row_cells(data_lines[0])]
    num_cols = max(len(r) for r in all_rows_parsed)

    # Flatten multi-level headers with '_'
    if len(header_rows) == 1:
        columns = header_rows[0]
    else:
        columns = []
        for col_idx in range(num_cols):
            parts = []
            prev = None
            for row in header_rows:
                val = row[col_idx].strip() if col_idx < len(row) else ''
                if val and val != prev:
                    parts.append(val)
                prev = val
            columns.append('_'.join(parts) if parts else '')

    # Replace spaces with underscores in column names
    columns = [c.replace(' ', '_') for c in columns]
    # Fix stray underscore in '@_N' → '@N'
    columns = [re.sub(r'@_(\d)', r'@\1', c) for c in columns]

    # Pad missing columns
    while len(columns) < num_cols:
        columns.append(f'col_{len(columns)}')

    # Parse data rows
    data_rows = []
    for line in data_lines:
        if not line.strip():
            continue
        cells = parse_row_cells(line)
        while len(cells) < num_cols:
            cells.append('')
        row = [coerce_value(c) for c in cells[:num_cols]]
        if not all(v in ('-', '') for v in row):
            data_rows.append(row)

    return columns, data_rows


print("Parsing functions loaded.")

## Main extraction function

In [ ]:
# HTML table parser (LightOnOCR emits tables in HTML, not markdown)

# LaTeX → Unicode map for Greek letters common in KGE papers
_GREEK_MAP = {
    r'\\alpha': 'α', r'\\beta': 'β', r'\\gamma': 'γ', r'\\delta': 'δ',
    r'\\epsilon': 'ε', r'\\zeta': 'ζ', r'\\eta': 'η', r'\\theta': 'θ',
    r'\\iota': 'ι', r'\\kappa': 'κ', r'\\lambda': 'λ', r'\\mu': 'μ',
    r'\\nu': 'ν', r'\\xi': 'ξ', r'\\pi': 'π', r'\\rho': 'ρ',
    r'\\sigma': 'σ', r'\\tau': 'τ', r'\\phi': 'φ', r'\\chi': 'χ',
    r'\\psi': 'ψ', r'\\omega': 'ω',
    r'\\Gamma': 'Γ', r'\\Delta': 'Δ', r'\\Lambda': 'Λ', r'\\Sigma': 'Σ',
    r'\\Phi': 'Φ', r'\\Omega': 'Ω',
}

def _strip_inline_formatting(text: str) -> str:
    """Strip LaTeX and normalize whitespace from an HTML cell/column value."""
    text = text.strip()
    # Inline math $...$ → content
    text = re.sub(r'\$([^$]+?)\$', r'\1', text)
    # Style wrappers: textbf / textit / text / mathbf / mathrm / mathit
    for cmd in ('textbf', 'textit', 'text', 'mathbf', 'mathrm', 'mathit'):
        text = re.sub(r'\\' + cmd + r'\{(.+?)\}', r'\1', text)
    # \frac{a}{b} → a/b
    text = re.sub(r'\\frac\{([^{}]+)\}\{([^{}]+)\}', r'\1/\2', text)
    # Drop \left / \right, keep the delimiter
    text = re.sub(r'\\(left|right)', '', text)
    # Common symbols
    text = text.replace('\\times', 'x')
    text = text.replace('\\cdot', '·')
    text = text.replace('\\pm', '±')
    text = text.replace('\\leq', '≤').replace('\\geq', '≥')
    # Greek letters (only if not followed by a letter, to avoid breaking e.g. \etak)
    for latex, uni in _GREEK_MAP.items():
        text = re.sub(latex + r'(?![a-zA-Z])', uni, text)
    # Typography escapes: \#, \&, \$, \%, \_
    text = re.sub(r'\\([#&$%_])', r'\1', text)
    # Simple sub/superscripts: _{xxx} or ^{xxx} → xxx (iterate for shallow nesting)
    for _ in range(3):
        text = re.sub(r'[_^]\{([^{}]*)\}', r'\1', text)
    # Leftover braces
    text = text.replace('{', '').replace('}', '')
    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text)
    # Trim spaces inside parentheses: '( D = 200 )' → '(D = 200)'
    text = re.sub(r'\(\s+', '(', text)
    text = re.sub(r'\s+\)', ')', text)
    return text.strip()


def extract_html_tables(text: str) -> list:
    """Return the list of <table>...</table> blocks from the OCR output."""
    return re.findall(r'<table\b[^>]*>.*?</table>', text, flags=re.DOTALL | re.IGNORECASE)


def _expand_row(cells, col_cursor_matrix, row_idx):
    """Expand a <tr> row respecting rowspan/colspan.
    col_cursor_matrix: dict {row_idx: {col_idx: value}} — filled in place."""
    col_idx = 0
    for cell in cells:
        # Advance to the first free column in this row
        while col_cursor_matrix.get(row_idx, {}).get(col_idx) is not None:
            col_idx += 1

        text = _strip_inline_formatting(cell.get_text(separator=' '))
        try:
            rowspan = int(cell.get('rowspan', 1))
        except (TypeError, ValueError):
            rowspan = 1
        try:
            colspan = int(cell.get('colspan', 1))
        except (TypeError, ValueError):
            colspan = 1

        for dr in range(rowspan):
            for dc in range(colspan):
                col_cursor_matrix.setdefault(row_idx + dr, {})[col_idx + dc] = text
        col_idx += colspan


def _matrix_to_rows(matrix: dict) -> list:
    """Convert the {row: {col: val}} matrix into a rectangular list of lists."""
    if not matrix:
        return []
    n_rows = max(matrix.keys()) + 1
    n_cols = max((max(cols.keys()) + 1) for cols in matrix.values() if cols)
    grid = []
    for r in range(n_rows):
        row = [matrix.get(r, {}).get(c, '') for c in range(n_cols)]
        grid.append(row)
    return grid


def parse_html_table(html_str: str) -> tuple:
    """Parse an HTML table into (columns, data_rows), matching the manual GT format:
    - Expands rowspan/colspan
    - Flattens multi-row headers with '_' between levels (no consecutive duplicates)
    - Fixes '@_N' → '@N'
    - Coerces values via coerce_value"""
    soup = BeautifulSoup(html_str, 'html.parser')
    table = soup.find('table')
    if table is None:
        return None, None

    # Separate matrices for thead and tbody
    header_matrix: dict = {}
    body_matrix: dict = {}

    thead = table.find('thead')
    tbody = table.find('tbody')

    if thead is not None:
        for r_idx, tr in enumerate(thead.find_all('tr')):
            cells = tr.find_all(['th', 'td'])
            _expand_row(cells, header_matrix, r_idx)

    if tbody is not None:
        body_trs = tbody.find_all('tr')
    else:
        # No explicit tbody: rows outside thead are data rows
        all_trs = table.find_all('tr')
        body_trs = [tr for tr in all_trs if (thead is None or tr not in thead.find_all('tr'))]

    for r_idx, tr in enumerate(body_trs):
        cells = tr.find_all(['th', 'td'])
        _expand_row(cells, body_matrix, r_idx)

    header_rows = _matrix_to_rows(header_matrix)
    body_rows = _matrix_to_rows(body_matrix)

    # No thead: use the first body row as header
    if not header_rows and body_rows:
        header_rows = [body_rows[0]]
        body_rows = body_rows[1:]

    if not header_rows or not body_rows:
        return None, None

    # Column count = max between header and body
    num_cols = max(
        max((len(r) for r in header_rows), default=0),
        max((len(r) for r in body_rows), default=0),
    )
    header_rows = [r + [''] * (num_cols - len(r)) for r in header_rows]
    body_rows = [r + [''] * (num_cols - len(r)) for r in body_rows]

    # Flatten multi-row headers with '_' BETWEEN levels (keep spaces within a level)
    if len(header_rows) == 1:
        columns = [_strip_inline_formatting(c) for c in header_rows[0]]
    else:
        columns = []
        for c in range(num_cols):
            parts = []
            prev = None
            for r in header_rows:
                val = _strip_inline_formatting(r[c])
                if val and val != prev:
                    parts.append(val)
                prev = val
            columns.append('_'.join(parts) if parts else '')

    # Fix stray '_' in '@_N' (Hits@_1 → Hits@1)
    columns = [re.sub(r'@_(\d)', r'@\1', c) for c in columns]
    # Empty header → '' (matches manual GT), not 'col_i'
    columns = [c if c else '' for c in columns]

    # Coerce data rows
    data_rows = []
    for row in body_rows:
        coerced = [coerce_value(v) for v in row]
        if not all(v in ('-', '') for v in coerced):
            data_rows.append(coerced)

    if not data_rows:
        return None, None

    return columns, data_rows


def extract_tables_from_pdf(pdf_path: Path) -> dict:
    """Full pipeline for one PDF:
    1. Render each page as an image
    2. Run LightOnOCR on each page
    3. Extract HTML (or markdown) tables from the OCR output
    4. Parse each table into columns + data rows
    5. Return a dict compatible with the manual GT structure."""
    paper_title = pdf_path.stem
    print(f"\n{'='*70}")
    print(f"Processing: {paper_title}")
    print(f"{'='*70}")

    pdf_doc = pdfium.PdfDocument(str(pdf_path))
    num_pages = len(pdf_doc)
    print(f"Pages: {num_pages}")

    all_tables = []  # (page_num_1based, columns, data_rows, raw_block)

    for page_idx in range(num_pages):
        page_num = page_idx + 1  # 1-indexed to match manual GT
        print(f"  Page {page_num}/{num_pages}...", end=" ", flush=True)

        pil_image = render_pdf_page(pdf_doc, page_idx)
        ocr_text = ocr_page(pil_image)

        # 1) Primary: HTML tables (what LightOnOCR usually emits)
        html_tables = extract_html_tables(ocr_text)
        # 2) Fallback: markdown pipe tables
        md_tables = extract_markdown_tables(ocr_text) if not html_tables else []

        total = len(html_tables) + len(md_tables)
        if total:
            print(f"→ {total} table(s)")
            for html_tbl in html_tables:
                columns, data_rows = parse_html_table(html_tbl)
                if columns and data_rows:
                    all_tables.append((page_num, columns, data_rows, html_tbl))
                    print(f"    HTML table → {len(data_rows)} rows × {len(columns)} cols")
            for md_tbl in md_tables:
                columns, data_rows = parse_markdown_table(md_tbl)
                if columns and data_rows:
                    all_tables.append((page_num, columns, data_rows, md_tbl))
                    print(f"    MD table → {len(data_rows)} rows × {len(columns)} cols")
        else:
            print("→ no tables")

    pdf_doc.close()

    # Build output JSON
    tables_json = []
    for i, (page_num, columns, data_rows, _) in enumerate(all_tables):
        table_id = f"table_{i + 1}"
        tables_json.append({
            "table_id": table_id,
            "page": page_num,
            "evaluation": {
                "expected_rows": len(data_rows),
                "expected_cols": len(columns),
                "columns": columns,
            },
            "rows": data_rows,
        })
        print(f"  [{table_id}] page {page_num} → {len(data_rows)} rows × {len(columns)} cols")
        print(f"    Columns: {columns}")

    document = {
        "paper_title": paper_title,
        "num_tables": len(tables_json),
        "tables": tables_json,
    }

    print(f"  → Total tables in ground truth: {len(tables_json)}")
    return document


print("Extraction function loaded.")

## Process the 5 test PDFs

In [ ]:
# Diagnostic: inspect raw OCR output on a page known to contain a table
# Default: "Binarized Knowledge Graph Embeddings" → page 8 (0-indexed → 7)
import time

if not PDF_FILES:
    raise RuntimeError(
        f"No PDFs in {PDF_DIR}. Re-run the paths cell and set PDF_DIR to your folder."
    )

DIAG_PDF_KEYWORD = "Binarized"  # change if your corpus uses another paper
DIAG_PAGE_IDX = 7               # page 8 (1-based)

matches = [p for p in PDF_FILES if DIAG_PDF_KEYWORD.lower() in p.name.lower()]
if matches:
    test_pdf = matches[0]
else:
    test_pdf = PDF_FILES[0]
    print(f"⚠ No PDF matching {DIAG_PDF_KEYWORD!r}; using first file instead.")

print(f"PDF: {test_pdf.name}")

pdf_doc = pdfium.PdfDocument(str(test_pdf))
n_pages = len(pdf_doc)
if DIAG_PAGE_IDX >= n_pages:
    DIAG_PAGE_IDX = max(0, n_pages // 2)
    print(f"⚠ Page 8 out of range ({n_pages} pages); using page {DIAG_PAGE_IDX + 1}.")

pil_img = render_pdf_page(pdf_doc, page_idx=DIAG_PAGE_IDX)
print(f"Rendered image: {pil_img.size} (page {DIAG_PAGE_IDX + 1}/{n_pages})")

t0 = time.time()
ocr_text = ocr_page(pil_img, max_new_tokens=2048)  # fewer tokens for a fast check
elapsed = time.time() - t0
pdf_doc.close()

print(f"OCR in {elapsed:.1f}s")
print("=" * 80)
print(ocr_text[:4000])
print("=" * 80)
print(
    f"Total chars: {len(ocr_text)} | lines: {len(ocr_text.splitlines())} "
    f"| with '|': {sum('|' in l for l in ocr_text.splitlines())}"
)

In [ ]:
documents = []

for pdf_path in PDF_FILES:
    doc = extract_tables_from_pdf(pdf_path)
    documents.append(doc)

ground_truth = {"documents": documents}

print(f"\n{'='*70}")
print("FINAL SUMMARY")
print(f"{'='*70}")
total_tables = sum(d['num_tables'] for d in documents)
for d in documents:
    print(f"  {d['paper_title'][:60]:60s} → {d['num_tables']} tables")
print(f"\nTotal tables extracted: {total_tables}")

## Save ground truth JSON

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(ground_truth, f, indent=2, ensure_ascii=False)

print(f"Ground truth saved to: {OUTPUT_PATH}")
print(f"Documents: {len(documents)}")
print(f"Total tables: {sum(d['num_tables'] for d in documents)}")

## Preview of the generated ground truth

In [ ]:
# Preview each document
for doc in ground_truth['documents']:
    print(f"\n{'─'*60}")
    print(f"Paper: {doc['paper_title']}")
    print(f"Tables: {doc['num_tables']}")
    for tbl in doc['tables']:
        print(f"\n  {tbl['table_id']} (page {tbl['page']})")
        print(f"  Dims: {tbl['evaluation']['expected_rows']} rows × {tbl['evaluation']['expected_cols']} cols")
        print(f"  Columns: {tbl['evaluation']['columns']}")
        if tbl['rows']:
            print(f"  First row: {tbl['rows'][0]}")
            if len(tbl['rows']) > 1:
                print(f"  Last row:  {tbl['rows'][-1]}")